### Rag with mongodb data integration , retrieval and Generation Pipeline

In [1]:
## Data ingestion and processing
import os
from dotenv import load_dotenv
from typing import TypedDict, List, Optional
from langgraph.graph import StateGraph, END
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

C:\Users\HP\AppData\Local\Temp\ipykernel_12080\2489423046.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
d:\ProjectsGit\ai-ml\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
##initialize the GEMINI API client
gemini_apikey = os.getenv("GEMINI_API_KEY")
llm= ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    temperature=0.2,
    google_api_key=gemini_apikey,
)

from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={"batch_size": 256, "normalize_embeddings": True},
)   

C:\Users\HP\AppData\Local\Temp\ipykernel_12080\1150027069.py:11: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4690.95it/s]


In [3]:
### function to generate embeddings for a given text
def get_embeddings(text,input_type="document"):
    response =  embeddings.embed_query(text)
    return response

In [4]:
get_embeddings("RAG technology")


[-0.11428561061620712,
 0.10915295779705048,
 0.038604043424129486,
 -0.03926149755716324,
 -0.14028409123420715,
 -0.014913646504282951,
 0.06698321551084518,
 0.06414292752742767,
 -0.09071972966194153,
 -0.005208243615925312,
 0.021872837096452713,
 0.03008168376982212,
 0.04258647561073303,
 -0.001258387346751988,
 -0.058476150035858154,
 0.04154346510767937,
 0.07989009469747543,
 0.07295897603034973,
 -0.012653354555368423,
 -0.034212078899145126,
 -0.04003624990582466,
 0.0040792981162667274,
 0.05261639505624771,
 -0.03245868533849716,
 0.029623309150338173,
 0.03343468904495239,
 -0.050022002309560776,
 -0.04280446842312813,
 0.06856043636798859,
 -0.04900268465280533,
 -0.04073643684387207,
 0.08663056790828705,
 -0.08248723298311234,
 -0.055238984525203705,
 -0.06547433882951736,
 -0.009302705526351929,
 0.0033360382076352835,
 0.05094841495156288,
 -0.0033073967788368464,
 0.013080154545605183,
 0.0018638877663761377,
 -0.06283721327781677,
 0.014675394631922245,
 -0.079018

In [5]:
### Data ingestion
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter   

#load the PDF document

loader = PyPDFLoader("https://arxiv.org/pdf/2307.09288.pdf")
data=loader.load()
##split the document into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=20)
documents= text_splitter.split_documents(data)


Exceeded 5000 form XObject invocations while extracting text; further form content is skipped.


In [6]:
documents

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'https://arxiv.org/pdf/2307.09288.pdf', 'total_pages': 77, 'page': 0, 'page_label': '1'}, page_content='Llama 2: Open Foundation and Fine-Tuned Chat Models\nHugo Touvron∗ Louis Martin† Kevin Stone†\nPeter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra\nPrajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen\nGuillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller'),
 Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:3

In [7]:
## Prepare Documents for insertion
docs_to_insert=[
    {
        "text": doc.page_content,
        "embedding": get_embeddings(doc.page_content),
        
    } for doc in documents
]

In [ ]:
docs_to_insert

In [8]:
from pymongo import MongoClient
#connect to the MongoDB database
client = MongoClient(os.getenv("mongodb_uri"))
collection = client["sample_mflix"]["ragpdf"]

#insert documents into the collection
result=collection.insert_many(docs_to_insert)
result

InsertManyResult([ObjectId('6aca2f5b238e34901ac91517'), ObjectId('6aca2f5b238e34901ac91518'), ObjectId('6aca2f5b238e34901ac91519'), ObjectId('6aca2f5b238e34901ac9151a'), ObjectId('6aca2f5b238e34901ac9151b'), ObjectId('6aca2f5b238e34901ac9151c'), ObjectId('6aca2f5b238e34901ac9151d'), ObjectId('6aca2f5b238e34901ac9151e'), ObjectId('6aca2f5b238e34901ac9151f'), ObjectId('6aca2f5b238e34901ac91520'), ObjectId('6aca2f5b238e34901ac91521'), ObjectId('6aca2f5b238e34901ac91522'), ObjectId('6aca2f5b238e34901ac91523'), ObjectId('6aca2f5b238e34901ac91524'), ObjectId('6aca2f5b238e34901ac91525'), ObjectId('6aca2f5b238e34901ac91526'), ObjectId('6aca2f5b238e34901ac91527'), ObjectId('6aca2f5b238e34901ac91528'), ObjectId('6aca2f5b238e34901ac91529'), ObjectId('6aca2f5b238e34901ac9152a'), ObjectId('6aca2f5b238e34901ac9152b'), ObjectId('6aca2f5b238e34901ac9152c'), ObjectId('6aca2f5b238e34901ac9152d'), ObjectId('6aca2f5b238e34901ac9152e'), ObjectId('6aca2f5b238e34901ac9152f'), ObjectId('6aca2f5b238e34901ac915

In [15]:
### query with search index 
from pymongo.operations import SearchIndexModel   
import time

#create a search index on the collection
index_name="vector_index"
search_index_model=SearchIndexModel(
    definition={
        "fields":[
            {
             "type":"vector",
             "numDimensions":384,
             "path":"embedding",
             "similarity": "cosine"  
            }
        ]
    },
    name=index_name,
    type="vectorSearch"
)
collection.create_search_index(search_index_model)

'vector_index'

In [16]:
##Query embedding
query_embedding=get_embeddings("RAG technology")
query_embedding

[-0.11428561061620712,
 0.10915295779705048,
 0.038604043424129486,
 -0.03926149755716324,
 -0.14028409123420715,
 -0.014913646504282951,
 0.06698321551084518,
 0.06414292752742767,
 -0.09071972966194153,
 -0.005208243615925312,
 0.021872837096452713,
 0.03008168376982212,
 0.04258647561073303,
 -0.001258387346751988,
 -0.058476150035858154,
 0.04154346510767937,
 0.07989009469747543,
 0.07295897603034973,
 -0.012653354555368423,
 -0.034212078899145126,
 -0.04003624990582466,
 0.0040792981162667274,
 0.05261639505624771,
 -0.03245868533849716,
 0.029623309150338173,
 0.03343468904495239,
 -0.050022002309560776,
 -0.04280446842312813,
 0.06856043636798859,
 -0.04900268465280533,
 -0.04073643684387207,
 0.08663056790828705,
 -0.08248723298311234,
 -0.055238984525203705,
 -0.06547433882951736,
 -0.009302705526351929,
 0.0033360382076352835,
 0.05094841495156288,
 -0.0033073967788368464,
 0.013080154545605183,
 0.0018638877663761377,
 -0.06283721327781677,
 0.014675394631922245,
 -0.079018

In [17]:
collection.ragpdf.aggregate([
    {
        "$vectorSearch":{
            "index":"vector_index",
            "queryVector":query_embedding,
            "path":"embedding",
            "numCandidates":384,
            "limit":5
        }
    }
])

In [18]:
#define a function to perform RAG with MongoDB
from pymongo import results


def get_query_results(query):
    """get results from MongoDB for a given query"""
    query_embedding = get_embeddings(query,input_type="query")
    print(f"Query embedding: {query_embedding}")
    pipeline=[
        {
            "$vectorSearch":{
                "index":"vector_index",
                "queryVector":query_embedding,
                "path":"embedding",
                "numCandidates":384,
                "limit":5
            }
        },{
            "$project":{
                "text":1,
                "_id": 0,
            }
        }
    ]
    results=collection.aggregate(pipeline)
    print(f"Results: {results}")
    
    
    array_of_results=[]
    for doc in results:
        array_of_results.append(doc)
    return array_of_results

In [19]:
get_query_results("Beyond Human Supervision")

Query embedding: [-0.013727838173508644, 0.015619908459484577, 0.00747190834954381, -0.0052528176456689835, -0.0014480127720162272, 0.0017341948114335537, 0.018445871770381927, 0.0085085928440094, 0.010249454528093338, -0.008242230862379074, 0.0234196987003088, -0.010683945380151272, -0.031702399253845215, 0.08330648392438889, 0.0005934980581514537, -0.04429898038506508, 0.045332588255405426, -0.04433774948120117, -0.038037728518247604, -0.11050264537334442, -0.025913693010807037, 0.011612283065915108, 0.0241752490401268, -0.02680325321853161, -0.07930584996938705, 0.033359233289957047, -0.04486235976219177, -0.129147008061409, 0.047040343284606934, -0.05857390910387039, 0.055806923657655716, -0.028869932517409325, 0.05251380056142807, -0.0137272784486413, -0.03482220694422722, 0.07114662975072861, 0.04891234636306763, 0.03662422299385071, 0.023973409086465836, -0.00974033772945404, -0.028226280584931374, -0.00028707171441055834, -0.03173784539103508, 0.010146692395210266, 0.0163578111

[{'text': 'LLMs, as manifested in surpassing human annotators in certain tasks, are fundamentally driven by RLHF, as\ndocumented in Gilardi et al. (2023) and Huang et al. (2023). Supervised data may no longer be the gold\nstandard, and this evolving circumstance compels a re-evaluation of the concept of “supervision.”\nIn-ContextTemperatureRescaling. WehaveobservedanintriguingphenomenonrelatedtoRLHF,afeature'},
 {'text': 'Fort, Deep Ganguli, Tom Henighan, et al. Training a helpful and harmless assistant with reinforcement\nlearning from human feedback.arXiv preprint arXiv:2204.05862, 2022a.\nYuntao Bai, Saurav Kadavath, Sandipan Kundu, Amanda Askell, Jackson Kernion, Andy Jones, Anna Chen,\nAnna Goldie, Azalia Mirhoseini, Cameron McKinnon, et al. Constitutional ai: Harmlessness from ai'},
 {'text': 'can be created by humans or by LLMs themselves (Zhou et al., 2022), and follow-up instructions can be used\nto refine initial generations to make them more useful, engaging, and unbiased (G

In [22]:
##llm
def get_answer_from_llm(query):
    ### embedding for the query
    context=get_query_results(query)
    context_string=" ".join([doc["text"] for doc in context])

    #prompt
    """get answer from LLM for a given query and context"""
    prompt=f"Answer the following question based on the context provided:\n\nContext: {context_string}\n\nQuestion: {query}"
    response=llm.invoke([("human", prompt)])
    return response.content


In [23]:
get_answer_from_llm("Beyond Human Supervision")

Query embedding: [-0.013727838173508644, 0.015619908459484577, 0.00747190834954381, -0.0052528176456689835, -0.0014480127720162272, 0.0017341948114335537, 0.018445871770381927, 0.0085085928440094, 0.010249454528093338, -0.008242230862379074, 0.0234196987003088, -0.010683945380151272, -0.031702399253845215, 0.08330648392438889, 0.0005934980581514537, -0.04429898038506508, 0.045332588255405426, -0.04433774948120117, -0.038037728518247604, -0.11050264537334442, -0.025913693010807037, 0.011612283065915108, 0.0241752490401268, -0.02680325321853161, -0.07930584996938705, 0.033359233289957047, -0.04486235976219177, -0.129147008061409, 0.047040343284606934, -0.05857390910387039, 0.055806923657655716, -0.028869932517409325, 0.05251380056142807, -0.0137272784486413, -0.03482220694422722, 0.07114662975072861, 0.04891234636306763, 0.03662422299385071, 0.023973409086465836, -0.00974033772945404, -0.028226280584931374, -0.00028707171441055834, -0.03173784539103508, 0.010146692395210266, 0.0163578111

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Results: <pymongo.synchronous.command_cursor.CommandCursor object at 0x000001F346B96450>


[{'type': 'text',
  'text': 'Based on the provided context, the section **"Beyond Human Supervision"** begins with:\n\n> *"At the outset of the project, many among us expressed a preference for"* (the text cuts off here in the excerpt).\n\nRelatedly, the context notes earlier that:\n* LLMs have begun surpassing human annotators in certain tasks, fundamentally driven by Reinforcement Learning from Human Feedback (RLHF).\n* Supervised data may no longer be considered the gold standard, leading to a necessary re-evaluation of the concept of "supervision."',
  'extras': {'signature': 'EowMCokMAWkUfRN4KQJxFORmFUBUk0NmyBu8EUHVC1TJyrLDWw1AOkNs++K8/u5AeZPii5kuEGF6TSgZdbTKANPxtUtxBpMeissBUUJwXEO2qTBlTua8GWsujYk+DjuUDlaIzoXJw+TViI4F4mujv4Qun7j1k4VLzUXwlPCPhB+oMoCdQFFGYbJdK2YAtppiASZAnsQbwKrWuBDj0oARh0Og1SomPxWaCyKSAUyHIIsO+rUASrmXsc0GMKE5YxZL50MKFOQVQkvUsVIyA9ZeMg3ckZAunsjk5jRMFkFChwy/gB8l0fed6IY1XU01FX1blFiJeG0xM0/O5So4DFJmu4ycVxJTtI26ihtl72GoxcMMzYXUnP+WPUQJZl1uZ30hArYiyzuOdEWJMDDG7PAACukP9WC8